# Portfolio Diversification — Buy & Hold with Risk Targeting

Tests the diversification benefit of combining several **unleveraged (≤1x)** buy-and-hold
strategies, each sized with volatility targeting. Every instrument is first sliced to the
common date range it shares with the others, so all strategies are measured over the **same period**.

In [ ]:
import sys
from pathlib import Path

# Locate the project root (the directory containing 'src')
project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
DATA_DIR = project_root / "data"
print(f"Project root: {project_root}")

from src.data import load_simple_price_csv
from src.core import Asset, Capital, FixedRiskSizer
from src.engine import BacktestRunner, PortfolioRunner
from src.strategies import BuyAndHoldStrategy
from src.visualization import print_comparison_table, print_portfolio_comparison, print_portfolio_diversification

## 1. Load instruments

`RGBITR1Y` is intentionally omitted — it is currently an exact duplicate of `RGBITR`.

In [ ]:
INSTRUMENTS = {
    "MCFTR":    "MCFTR.csv",      # broad equity index
    "RGBITR":   "RGBITR.csv",     # government bond index (total return)
    "GLDRUB":   "GLDRUB_TOM.csv", # gold, in RUB
    "CNYRUB":   "CNYRUB_TOM.csv", # CNY/RUB FX
    "USDRUB":   "USDRUB.csv",     # USD/RUB FX
}

assets = {
    ticker: Asset(
        ticker=ticker,
        price_data=load_simple_price_csv(DATA_DIR / filename),
        commission_rate=0.0004,
        slippage_rate=0.001,
    )
    for ticker, filename in INSTRUMENTS.items()
}

## 2. Slice to the common period

Each instrument has a different history (MCFTR starts in 2003, gold/CNYRUB in 2013, USDRUB in 1992).
To compare them fairly, we restrict everything to the date range they all share, using the new
`Asset.slice()` helper (which preserves point value, commission, slippage, etc.).

In [ ]:
start = max(a.price_data.index.min() for a in assets.values())
end   = min(a.price_data.index.max() for a in assets.values())
print(f"Common period: {start.date()} -> {end.date()}")

assets = {ticker: a.slice(start, end) for ticker, a in assets.items()}

for ticker, a in assets.items():
    print(f"  {ticker:8s} {a.price_data.index.min().date()} -> {a.price_data.index.max().date()}  ({len(a.price_data):,} rows)")

## 3. Buy & hold with risk targeting

Volatility targeting sizes each position so its annualised risk tends toward the target,
capped at 100% notional (`max_leverage=1.0`) since these are unleveraged index/spot instruments.

In [ ]:
INITIAL_CAPITAL = 100_000
RISK_TARGET = 0.20     # 20% annualised volatility target
MAX_LEVERAGE = 1.0     # no leverage

capital = Capital(initial_capital=INITIAL_CAPITAL)
sizer = FixedRiskSizer(risk_target=RISK_TARGET, max_leverage=MAX_LEVERAGE)

reports = {
    ticker: BacktestRunner(capital, asset, sizer).run(BuyAndHoldStrategy())
    for ticker, asset in assets.items()
}

## 4. Individual strategies (same period)

All strategies now span the same ~12.3 years, so their metrics are directly comparable.

In [ ]:
print_comparison_table(
    "Individual B&H — same period, risk target 20%, ≤1x leverage",
    {ticker: r.metrics for ticker, r in reports.items()},
)

## 5. Portfolio + diversification

Combine the strategies at equal weight and inspect the diversification benefit:
the correlation matrix, the diversification ratio, and the volatility reduction.

In [ ]:
# Equal-weight portfolio (weights=None -> 1/N each)
portfolio = PortfolioRunner(capital).run(reports, weights=None)

print_portfolio_diversification(portfolio.result)
print_portfolio_comparison(portfolio.metrics, {ticker: r.metrics for ticker, r in reports.items()})